# Date Palm Leaf Disease Detection: Training Notebook

Classifies a date palm leaf photo into **Healthy**, **Brown Spot** or **White Scale** with a ResNet18, then exports the model to ONNX for the FastAPI app.

**Steps:** setup, near-duplicate grouping and group-wise split, shortcut check, extra data, training, evaluation, Grad-CAM, ONNX export.

**Data (not included, download from Kaggle):**
- Date Palm Data: https://www.kaggle.com/datasets/hadjerhamaidi/date-palm-data
- White Cochineal: https://www.kaggle.com/datasets/bouthinarechachi/1-white-cochiniel-disease
- Own photos: 31 natural-background healthy leaves (`healthy.zip`) and 35 White Scale leaves (`white_extra.zip`)

**Notes**
- Training has randomness, so a rerun gives slightly different numbers. The notebook saves to new file names so it never overwrites the model used by the app.
- The split is saved to `split_v3.csv`. If the file exists it is loaded, so the test set stays the same.
- Use a GPU runtime (Runtime, Change runtime type, T4 GPU).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q imagehash pillow-avif-plugin onnx onnxruntime
!unzip -q -o "/content/drive/MyDrive/Date Palm data.zip" -d /content/data
!unzip -q -o "/content/drive/MyDrive/healthy.zip" -d /content/healthy_nat
!unzip -q -o "/content/drive/MyDrive/white_extra.zip" -d /content/white_extra
!unzip -q -o -j "/content/drive/MyDrive/white_ext.zip" -d /content/white_ext

In [ ]:
import glob, os, shutil, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, pillow_avif
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

dev = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["brown spots", "healthy", "white scale"]
OK = (".jpg", ".jpeg", ".png", ".webp", ".jfif", ".bmp", ".avif")
SPLIT_CSV = "/content/drive/MyDrive/split_v3.csv"
MODEL_OUT = "/content/drive/MyDrive/palm_model_v6_rerun.pt"      # new name: never overwrites the app model
ONNX_OUT = "/content/drive/MyDrive/palm_model_v6_rerun.onnx"

def imgs(folder):
    return sorted(f for f in glob.glob(f"{folder}/**/*", recursive=True)
                  if os.path.isfile(f) and "__MACOSX" not in f and f.lower().endswith(OK))

## 2. Near-duplicate groups and group-wise split
The dataset contains flipped and resized copies of the same photos. Copies are grouped (perceptual hash, also on the mirrored image) and each group stays in one split. A plain random split put copies in both train and test and gave a false 100% accuracy.

In [ ]:
import imagehash
from PIL import ImageOps
from sklearn.model_selection import GroupShuffleSplit

if os.path.exists(SPLIT_CSV):
    df = pd.read_csv(SPLIT_CSV)
    print("Loaded saved split:", len(df), "images")
else:
    SRC = "/content/data/Date Palm data"
    paths, labels, h1, h2 = [], [], [], []
    for c in sorted(os.listdir(SRC)):
        for f in sorted(os.listdir(f"{SRC}/{c}")):
            p = f"{SRC}/{c}/{f}"
            try:
                im = Image.open(p).convert("RGB")
            except Exception:
                continue
            paths.append(p); labels.append(c)
            h1.append(int(str(imagehash.phash(im)), 16))
            h2.append(int(str(imagehash.phash(ImageOps.mirror(im))), 16))

    par = list(range(len(paths)))                      # union-find
    def find(i):
        while par[i] != i:
            par[i] = par[par[i]]; i = par[i]
        return i
    T = 6                                              # max hash distance for "same image"
    for i in range(len(paths)):
        for j in range(i + 1, len(paths)):
            if min((h1[i] ^ h1[j]).bit_count(), (h1[i] ^ h2[j]).bit_count()) <= T:
                par[find(i)] = find(j)
    groups = [find(i) for i in range(len(paths))]
    print("Images:", len(paths), "| Unique groups:", len(set(groups)))

    idx = list(range(len(paths)))
    tr_i, rest_i = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42).split(idx, labels, groups))
    g_rest = [groups[i] for i in rest_i]
    v_k, t_k = next(GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42).split(rest_i, groups=g_rest))
    df = pd.DataFrame({"path": paths, "label": labels, "group": groups})
    df["split"] = "train"
    df.loc[[rest_i[k] for k in v_k], "split"] = "val"
    df.loc[[rest_i[k] for k in t_k], "split"] = "test"
    df.to_csv(SPLIT_CSV, index=False)

print(df.groupby(["split", "label"]).size())

## 3. Shortcut check
The healthy photos are much smaller than the other classes. A decision tree that sees only image size and mean color shows how much of the task can be solved without looking at the disease.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

feats = []
for p in df.path:
    im = Image.open(p).convert("RGB"); a = np.asarray(im)
    feats.append([im.size[0], im.size[1], a[..., 0].mean(), a[..., 1].mean(), a[..., 2].mean()])
X = np.array(feats); y = df.label.values
stats = pd.DataFrame(X, columns=["width", "height", "r", "g", "b"]); stats["label"] = y
stats["brightness"] = stats[["r", "g", "b"]].mean(axis=1)
print(stats.groupby("label")[["width", "height", "brightness"]].mean().round(1))

tr_m, te_m = (df.split == "train").values, (df.split == "test").values
clf = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X[tr_m], y[tr_m])
print("Test accuracy using only size + color:", round((clf.predict(X[te_m]) == y[te_m]).mean(), 3))

## 4. Extra data
- 31 healthy leaf photos with natural backgrounds (8 test, 4 validation, rest train)
- 35 White Scale photos taken by the team (about a quarter for test, rest train)
- 100 White Scale images from the White Cochineal dataset (70 train, 30 test)

In [ ]:
for c in ["natural", "extra", "ext"]:
    df[c] = False
rows = []
nat = imgs("/content/healthy_nat"); n = len(nat)
rows += [dict(path=p, label="healthy", group=f"nat{i}", natural=True, extra=False, ext=False,
              split="test" if i >= n - 8 else "val" if i >= n - 12 else "train") for i, p in enumerate(nat)]
ex = imgs("/content/white_extra"); k = len(ex); kt = max(4, k // 4)
rows += [dict(path=p, label="white scale", group=f"ex{i}", natural=False, extra=True, ext=False,
              split="test" if i >= k - kt else "train") for i, p in enumerate(ex)]
wx = imgs("/content/white_ext"); w = len(wx)
rows += [dict(path=p, label="white scale", group=f"wx{i}", natural=False, extra=False, ext=True,
              split="test" if i >= int(w * 0.7) else "train") for i, p in enumerate(wx)]
df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)
print("natural healthy:", n, "| own White Scale:", k, "| other-dataset White Scale:", w)
print(df.groupby(["split", "label"]).size())

## 5. Data loading
Training blurs a random half of the images (for every class) so that image sharpness cannot reveal the class. Natural-background healthy photos and our own White Scale photos are shown 3 times as often.

In [ ]:
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tr = transforms.Compose([
    transforms.RandomApply([transforms.Resize((112, 112))], p=0.5), transforms.Resize((224, 224)),
    transforms.RandomResizedCrop(224, scale=(0.3, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4), transforms.GaussianBlur(5),
    transforms.ToTensor(), norm, transforms.RandomErasing(p=0.5)])
ev224 = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])
ev112 = transforms.Compose([transforms.Resize((112, 112)), transforms.Resize((224, 224)),
                            transforms.ToTensor(), norm])

class DS(Dataset):
    def __init__(self, d, tf):
        self.p, self.tf = d.path.tolist(), tf
        self.y = [CLASSES.index(l) for l in d.label]
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

trd, vad, ted = (df[df.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
sampler = WeightedRandomSampler(np.where(trd.natural | trd.extra, 3.0, 1.0), num_samples=len(trd), replacement=True)
trl = DataLoader(DS(trd, tr), batch_size=32, sampler=sampler, num_workers=2)

def run(model, tf, d):
    """Softmax probabilities and true labels for a dataframe of images."""
    dl = DataLoader(DS(d, tf), batch_size=32, num_workers=2)
    pr, Y = [], []
    model.eval()
    with torch.no_grad():
        for x, y in dl:
            pr.append(torch.softmax(model(x.to(dev)), 1).cpu().numpy()); Y += y.tolist()
    return np.concatenate(pr), np.array(Y)

## 6. Training (ResNet18, a missed White Scale costs 1.5 times more)

In [ ]:
torch.manual_seed(42)
m = models.resnet18(weights="IMAGENET1K_V1")
m.fc = nn.Linear(m.fc.in_features, 3); m = m.to(dev)
opt = torch.optim.Adam(m.parameters(), 1e-4)
lossf = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.0, 1.5]).to(dev))

for ep in range(20):
    m.train()
    for x, y in trl:
        opt.zero_grad(); lossf(m(x.to(dev)), y.to(dev)).backward(); opt.step()
    pr, Y = run(m, ev224, vad)
    print(f"epoch {ep + 1}: val acc {(pr.argmax(1) == Y).mean():.3f}")

torch.save(m.state_dict(), MODEL_OUT)
print("saved", MODEL_OUT)

## 7. Evaluation on the test set

In [ ]:
pr, Y = run(m, ev224, ted); P = pr.argmax(1)
print("errors:", (P != Y).sum(), "/", len(Y))
print(classification_report(Y, P, target_names=CLASSES, digits=3))
print(confusion_matrix(Y, P))

nt, exm, xt = ted.natural.values, ted.extra.values, ted.ext.values
print("White Scale -> Healthy:", ((Y == 2) & (P == 1)).sum(),
      "| natural healthy:", (P[nt] == Y[nt]).sum(), "/", nt.sum(),
      "| own White Scale:", (P[exm] == Y[exm]).sum(), "/", exm.sum(),
      "| other-dataset White Scale:", (P[xt] == Y[xt]).sum(), "/", xt.sum())

ConfusionMatrixDisplay.from_predictions(Y, P, display_labels=CLASSES, cmap="Blues")
plt.savefig("/content/drive/MyDrive/confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

### Optional: comparison with the earlier model (v4)
Runs only if `palm_model_v4.pt` is in Drive. v4 was trained with every image blurred (112 then 224).

In [ ]:
V4 = "/content/drive/MyDrive/palm_model_v4.pt"
if os.path.exists(V4):
    v4 = models.resnet18(weights=None); v4.fc = nn.Linear(v4.fc.in_features, 3)
    v4.load_state_dict(torch.load(V4, map_location="cpu")); v4 = v4.to(dev).eval()
    pr4, _ = run(v4, ev112, ted); P4 = pr4.argmax(1)
    for name, PP in [("v4", P4), ("v6 (this run)", P)]:
        print(f"{name:14s} errors {(PP != Y).sum():3d}/{len(Y)} | White Scale -> Healthy {((Y == 2) & (PP == 1)).sum()}"
              f" | natural healthy {(PP[nt] == Y[nt]).sum()}/{nt.sum()} | own White Scale {(PP[exm] == Y[exm]).sum()}/{exm.sum()}")
else:
    print("v4 weights not found, skipping the comparison.")

## 8. Grad-CAM: where the model looks

In [ ]:
acts, grads = {}, {}
def fwd(mod, i, o):
    acts["a"] = o
    o.register_hook(lambda g: grads.update(g=g))
hook = m.layer4[-1].register_forward_hook(fwd)

def cam(path):
    img = Image.open(path).convert("RGB")
    x = ev224(img).unsqueeze(0).to(dev)
    m.eval(); m.zero_grad()
    out = m(x); c = out.argmax(1).item()
    out[0, c].backward()
    w = grads["g"].mean(dim=(2, 3), keepdim=True)
    cm = F.relu((w * acts["a"]).sum(1, keepdim=True))
    cm = F.interpolate(cm, size=(224, 224), mode="bilinear")[0, 0].detach().cpu().numpy()
    cm = (cm - cm.min()) / (cm.max() - cm.min() + 1e-8)
    return img.resize((224, 224)), cm, CLASSES[c]

fig, ax = plt.subplots(3, 4, figsize=(12, 9))
for r, cls in enumerate(CLASSES):
    for kk, p in enumerate(ted[ted.label == cls].path.tolist()[:2]):
        img, cm, pred = cam(p)
        ax[r, 2 * kk].imshow(img); ax[r, 2 * kk].set_title(cls)
        ax[r, 2 * kk + 1].imshow(img); ax[r, 2 * kk + 1].imshow(cm, cmap="jet", alpha=0.45)
        ax[r, 2 * kk + 1].set_title(f"model: {pred}")
for a in ax.ravel(): a.axis("off")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/gradcam.png", dpi=200, bbox_inches="tight")
plt.show()
hook.remove()

## 9. Export to ONNX for the app
The app runs on ONNX Runtime. The export returns the class scores and a class-activation map for every class, so the app can draw the heatmap without gradients.

In [ ]:
import onnxruntime as ort

m2 = models.resnet18(weights=None); m2.fc = nn.Linear(m2.fc.in_features, 3)
m2.load_state_dict(torch.load(MODEL_OUT, map_location="cpu")); m2.eval()

class Wrap(nn.Module):
    def __init__(self, mm):
        super().__init__(); self.m = mm
    def forward(self, x):
        mm = self.m
        x = mm.maxpool(mm.relu(mm.bn1(mm.conv1(x))))
        x = mm.layer4(mm.layer3(mm.layer2(mm.layer1(x))))
        return mm.fc(x.mean(dim=(2, 3))), torch.einsum("bkhw,ck->bchw", x, mm.fc.weight)

torch.onnx.export(Wrap(m2).eval(), torch.randn(1, 3, 224, 224), ONNX_OUT,
                  input_names=["input"], output_names=["logits", "cams"], opset_version=17, dynamo=False)
print("ONNX file:", round(os.path.getsize(ONNX_OUT) / 1e6, 1), "MB")

# the ONNX model should give the same test errors as the PyTorch model
sess = ort.InferenceSession(ONNX_OUT, providers=["CPUExecutionProvider"])
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32); STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
def prep(path):
    im = Image.open(path).convert("RGB").resize((224, 224), Image.BILINEAR)
    a = (np.asarray(im, dtype=np.float32) / 255 - MEAN) / STD
    return a.transpose(2, 0, 1)[None].astype(np.float32)
Po = np.array([sess.run(None, {"input": prep(p)})[0][0].argmax() for p in ted.path])
print("ONNX errors:", (Po != Y).sum(), "/", len(Y), "| agree with PyTorch on", (Po == P).mean().round(3), "of images")